# NB8 — Deployment

## Overview

Deploy YOLOv8m Ensemble Distilled model using two approaches:

| Approach | Platform | Purpose |
|----------|----------|---------|
| 1 | FastAPI + Docker + Google Cloud Run | Production REST API |
| 2 | Ultralytics HUB | Visual demo for anyone |

## Stack
- **Model:** YOLOv8m Ensemble Distilled (mAP@0.5: 0.866, Speed: 4.8ms)
- **API:** FastAPI + Uvicorn
- **Container:** Docker
- **Cloud:** Google Cloud Run (us-central1)
- **Visual Demo:** Ultralytics HUB (Las Vegas)

## Architecture
```
Phone/Browser
      ↓
HTTP POST (image)
      ↓
FastAPI on Google Cloud Run
      ↓
YOLOv8m Distilled (0.866 mAP, 4.8ms)
      ↓
JSON response {boxes, labels, scores}
```


## Approach 1 — FastAPI + Google Cloud Run

Production-grade REST API deployed as a serverless container.

### Prerequisites (run once on Mac)
```bash
# Install Homebrew
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

# Add to PATH
echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zshrc
eval "$(/opt/homebrew/bin/brew shellenv)"

# Install Docker and gcloud
brew install --cask docker
brew install --cask google-cloud-sdk

# Add gcloud to PATH
echo 'export PATH=/opt/homebrew/share/google-cloud-sdk/bin:"$PATH"' >> ~/.zshrc
source ~/.zshrc

# Authenticate
gcloud auth login
gcloud config set project solar-cycle-487619-t6
gcloud auth application-default login --scopes="https://www.googleapis.com/auth/cloud-platform,https://www.googleapis.com/auth/accounts.reauth"
gcloud auth application-default set-quota-project solar-cycle-487619-t6
gcloud auth configure-docker
```


In [1]:
# ============================================================
# Create FastAPI app — main.py
# Receives image → runs YOLOv8m Distilled → returns boxes
# ============================================================

main_py = '''
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO
from PIL import Image
import numpy as np
import io

app = FastAPI(title="Waymo Perception API")

# Allow all origins for phone camera demo
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model once at startup
model = YOLO("waymo_yolov8m_distilled_best.pt")

CLASS_NAMES = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

@app.get("/")
def health_check():
    return {"status": "ok", "model": "YOLOv8m Ensemble Distilled", "mAP50": 0.866}

@app.post("/detect")
async def detect(file: UploadFile = File(...)):
    # Read image
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Run inference
    results = model.predict(image, conf=0.35, verbose=False)[0]

    # Parse detections
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append({
            "class_id":   int(box.cls),
            "class_name": CLASS_NAMES[int(box.cls)],
            "confidence": round(float(box.conf), 3),
            "bbox": {
                "x1": round(x1, 1),
                "y1": round(y1, 1),
                "x2": round(x2, 1),
                "y2": round(y2, 1)
            }
        })

    return JSONResponse({
        "detections": detections,
        "count": len(detections),
        "image_size": {"width": image.width, "height": image.height}
    })
'''

# Save to file
with open("/content/main.py", "w") as f:
    f.write(main_py)
print("main.py created ✅")
print(open("/content/main.py").read())

main.py created ✅

from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO
from PIL import Image
import numpy as np
import io

app = FastAPI(title="Waymo Perception API")

# Allow all origins for phone camera demo
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model once at startup
model = YOLO("waymo_yolov8m_distilled_best.pt")

CLASS_NAMES = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

@app.get("/")
def health_check():
    return {"status": "ok", "model": "YOLOv8m Ensemble Distilled", "mAP50": 0.866}

@app.post("/detect")
async def detect(file: UploadFile = File(...)):
    # Read image
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Run inference
    results = model.predict(image, conf=0.35, verbose=False)[0]

    # Parse detections
    detecti

In [2]:
# ============================================================
# Create requirements.txt
# ============================================================

requirements = """fastapi==0.115.0
uvicorn==0.30.0
ultralytics==8.4.38
pillow==10.4.0
python-multipart==0.0.9
numpy==1.26.4
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt created ✅")
print(open("/content/requirements.txt").read())

requirements.txt created ✅
fastapi==0.115.0
uvicorn==0.30.0
ultralytics==8.4.38
pillow==10.4.0
python-multipart==0.0.9
numpy==1.26.4



In [3]:
# ============================================================
# Create Dockerfile
# ============================================================

dockerfile = """FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app and model
COPY main.py .
COPY waymo_yolov8m_distilled_best.pt .

# Run FastAPI
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
"""

with open("/content/Dockerfile", "w") as f:
    f.write(dockerfile)
print("Dockerfile created ✅")
print(open("/content/Dockerfile").read())

Dockerfile created ✅
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app and model
COPY main.py .
COPY waymo_yolov8m_distilled_best.pt .

# Run FastAPI
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]



In [7]:
# ============================================================
# Save all deployment files to GCS so they survive session crashes
# ============================================================
import subprocess

GCS_DEPLOY_PATH = "gs://mywaymo-perdataset-2026/deployment"

# Upload all files
for file in ["main.py", "requirements.txt", "Dockerfile"]:
    subprocess.run([
        "gsutil", "cp", f"/content/{file}",
        f"{GCS_DEPLOY_PATH}/{file}"
    ])
    print(f"✅ {file} saved to GCS")

# Verify
result = subprocess.run(
    ["gsutil", "ls", f"{GCS_DEPLOY_PATH}/"],
    capture_output=True, text=True
)
print(f"\nFiles in GCS:\n{result.stdout}")

✅ main.py saved to GCS
✅ requirements.txt saved to GCS
✅ Dockerfile saved to GCS

Files in GCS:
gs://mywaymo-perdataset-2026/deployment/Dockerfile
gs://mywaymo-perdataset-2026/deployment/main.py
gs://mywaymo-perdataset-2026/deployment/requirements.txt



In [6]:
# ============================================================
# GCS Authentication (Permanent — Service Account)
# ============================================================

#subprocess — Python run shell commands (like gcloud, gsutil) from inside Python
#threading — Python run a background task simultaneously while training runs
#time — used for time.sleep() (pause) and time.strftime() (print current time)
#os — used for file checks like os.path.exists()

import subprocess, threading, time, os

#below 3 variable definitions — store our paths/IDs
KEY_FILE   = "/content/gcs-key.json"
PROJECT_ID = "solar-cycle-487619-t6"
GCS_BUCKET = "gs://mywaymo-perdataset-2026"

#Defines a reusable function that re-authenticates GCS
def _reauth():
    # tells gcloud "use this service account key for all future GCS operations"
    subprocess.run(["gcloud", "auth", "activate-service-account",
                    "--key-file", KEY_FILE], capture_output=True)
    #— tells gcloud which GCP project to bill/access.
    #capture_output=True — suppresses output so it runs silently in the background without cluttering our notebook
    subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
                    capture_output=True)
# calls _reauths() to reauthenticate every 45 minutes by default (tokens expire after ~60 min, so 45 is safe).
#waits 45 minutes (45 × 60 = 2700 seconds)
def start_auth_keepalive(interval_minutes=45):
    #runs forever in a loop
    def _loop():
        while True:
            time.sleep(interval_minutes * 60)
            #silently refreshes the GCS token
            _reauth()
            #shows a timestamp so we know re-auth happened (useful for debugging)
            print(f"🔄 GCS re-auth at {time.strftime('%H:%M:%S')}")
    #threading.Thread(target=_loop) — creates a background thread that runs _loop
    #daemon=True->if the main (colab) program dies, kill this helper (_reauth() thread) too automatically."
    #.start() — start background thread immediately
    threading.Thread(target=_loop, daemon=True).start()
    #print confirms it started
    print(f"🔄 Keepalive started (every {interval_minutes} min)")

# Step 1 — bootstrap: needed interactive login just to grab the key once per session, then we immediately switch to the service account
!gcloud auth login --no-launch-browser #interactive one-time login — opens a URL,paste a code — this auth needed to download the key
#downloads gcs-key.json from my GCS bucket to /content/ on the Colab VM
!gsutil cp gs://mywaymo-perdataset-2026/auth/gcs-key.json /content/gcs-key.json

# Step 2 — immediately switches from the short-lived interactive login to service account (never expires)
_reauth()
start_auth_keepalive()#starts the background thread that re-auths every 45 min for the rest of the session

# Step 3 — verify

#Runs gsutil ls gs://mywaymo-perdataset-2026 to test the connection
#capture_output=True — captures the output instead of printing it directly
#text=True — returns output as a string (not bytes)
#result.returncode == 0 — 0 means success in Linux/shell, anything else is an error
#Prints ✅ if it worked, or the actual error message if it didn't

result = subprocess.run(["gsutil", "ls", GCS_BUCKET], capture_output=True, text=True)
print("✅ GCS authenticated and connected" if result.returncode == 0 else f"❌ {result.stderr}")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=ygjK6GZUoTcuCzGmo5heF2Kv95Snku&prompt=consent&token_usage=remote&access_type=offline&code_challenge=EJgOPiupMWBy_jG8f0OIY8CZVpP9hQVuuUf60lYuORw&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E9a-nMpMOJ0B0uhJef9Z2zj4jqNLSrFYnFQTrhxL8fzY80SkFF8CHGJB9Xk1t4pWw

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

### Build and Deploy (run in Mac Terminal)

```bash
# Create deployment folder and download files from GCS
mkdir ~/waymo_deployment
gsutil cp gs://mywaymo-perdataset-2026/deployment/main.py ~/waymo_deployment/
gsutil cp gs://mywaymo-perdataset-2026/deployment/requirements.txt ~/waymo_deployment/
gsutil cp gs://mywaymo-perdataset-2026/deployment/Dockerfile ~/waymo_deployment/
gsutil cp gs://mywaymo-perdataset-2026/models/waymo_yolov8m_distilled_best.pt ~/waymo_deployment/

# Enable GCP APIs
gcloud services enable run.googleapis.com
gcloud services enable containerregistry.googleapis.com

# Build Docker image (AMD64 for Cloud Run)
cd ~/waymo_deployment
docker build --platform linux/amd64 -t gcr.io/solar-cycle-487619-t6/waymo-perception:v1 .

# Test locally before deploying
docker run --platform linux/amd64 -p 8080:8080 gcr.io/solar-cycle-487619-t6/waymo-perception:v1
curl http://localhost:8080/

# Push to Google Container Registry
docker push gcr.io/solar-cycle-487619-t6/waymo-perception:v1

# Deploy to Cloud Run
gcloud run deploy waymo-perception \
  --image gcr.io/solar-cycle-487619-t6/waymo-perception:v1 \
  --platform managed \
  --region us-central1 \
  --allow-unauthenticated \
  --memory 4Gi \
  --cpu 2 \
  --port 8080 \
  --timeout 300
```


### Key Fixes During Build

- Build must use `--platform linux/amd64` — Cloud Run does not support ARM64 (M2 Mac)
- `python:3.11-slim` needs system libs for OpenCV:
  ```
  libxcb1 libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 libgl1
  ```
- Use `libgl1` not `libgl1-mesa-glx` (not available in Debian trixie)
- Use `--no-cache` when Dockerfile changes to avoid stale cached layers


In [ ]:
#Test the detect endpoint with a real image:

# Download a test image or from internet
curl -o test.jpg https://ultralytics.com/images/bus.jpg

# Send to your API i.e  Run this in Terminal
curl -X POST \
  https://waymo-perception-725477696855.us-central1.run.app/detect \
  -F "file=@test.jpg"

 # or
  curl -X POST \
  https://waymo-perception-725477696855.us-central1.run.app/detect \
  -F "file=@/Users/suhasini/Downloads/artisticrealms-vehicles-1-.jpg"
  #The Docker container is no longer running locally — our API is now live on Cloud Run permanently.

  #output:

  Last login: Sun Apr 19 11:12:20 on ttys001
(base) suhasini@Suhasinis-MacBook-Pro ~ % curl -X POST \
  https://waymo-perception-725477696855.us-central1.run.app/detect \
  -F "file=@/Users/suhasini/Downloads/artisticrealms-vehicles-1-.jpg"
{"detections":[{"class_id":0,"class_name":"Vehicle","confidence":0.851,"bbox":{"x1":3862.8,"y1":1867.6,"x2":5171.4,"y2":3300.4}},{"class_id":0,"class_name":"Vehicle","confidence":0.65,"bbox":{"x1":1907.5,"y1":1853.2,"x2":2203.2,"y2":2198.7}},{"class_id":0,"class_name":"Vehicle","confidence":0.536,"bbox":{"x1":46.0,"y1":1907.9,"x2":324.9,"y2":2224.9}},{"class_id":0,"class_name":"Vehicle","confidence":0.484,"bbox":{"x1":3062.0,"y1":1725.4,"x2":3437.8,"y2":2160.4}},{"class_id":0,"class_name":"Vehicle","confidence":0.379,"bbox":{"x1":96.4,"y1":1841.9,"x2":449.8,"y2":2291.0}}],"count":5,"image_size":{"width":5184,"height":3456}}%        (base) suhasini@Suhasinis-MacBook-Pro ~ %

#our API is fully working:
✅ Receives image
✅ Runs YOLOv8m Distilled (0.866 mAP)
✅ Returns JSON with boxes + confidence
✅ Live at permanent public URL

### API Response Example

```json
{
  "detections": [
    {"class_name": "Vehicle", "confidence": 0.851, "bbox": {"x1": 3862.8, "y1": 1867.6, "x2": 5171.4, "y2": 3300.4}},
    {"class_name": "Vehicle", "confidence": 0.650, "bbox": {"x1": 1907.5, "y1": 1853.2, "x2": 2203.2, "y2": 2198.7}}
  ],
  "count": 5,
  "image_size": {"width": 5184, "height": 3456}
}
```

**Cloud Run API URL:**
```
https://waymo-perception-725477696855.us-central1.run.app
```


## Approach 2 — Ultralytics HUB Visual Demo

Easiest way to get a visual demo — upload `.pt` model and get instant demo page.

### Steps
1. Go to **hub.ultralytics.com**
2. Sign in with Google
3. Create project → **Train** → Upload model
4. Upload `waymo_yolov8m_distilled_best.pt`
5. Click **Deploy** → Select region (Las Vegas — closest to Phoenix AZ)
6. Click **Deploy Model**
7. Click **Predict** tab → upload image → see visual detections

### Why Ultralytics HUB
- Natively supports `.pt` YOLO files
- Visual demo with bounding boxes drawn on image
- Shareable public URL — works on any device
- No coding needed
- Free tier available

### Detection Results
```
Vehicle    87.5% ✅
Vehicle    78.5% ✅
Cyclist    50.5% ✅
Pedestrian 53.5% ✅
Pedestrian 37.6% ✅
```

**Ultralytics HUB Demo URL:**
```
https://predict-69e551e576657ed89ece-dproatj77a-wn.a.run.app
```


## Deployment Summary

| Platform | URL | Purpose |
|----------|-----|---------|
| Cloud Run API | https://waymo-perception-725477696855.us-central1.run.app | Production REST API ✅ |
| Ultralytics HUB | https://predict-69e551e576657ed89ece-dproatj77a-wn.a.run.app | Visual Demo ✅ |

**Endpoints:**
```
GET  /         → health check + model info
POST /detect   → send image → get JSON detections
```


## License Notice

This model was trained on the **Waymo Open Dataset**, licensed for non-commercial use under the [Waymo Dataset License Agreement](https://waymo.com/open/terms/).

Any use or modification of this model is subject to the same non-commercial restrictions.

- ✅ Academic use allowed
- ✅ Portfolio/research demo allowed
- ❌ Commercial use not allowed
- ❌ Deployment in real vehicles not allowed


## Lessons Learned — Deployment

### Mac Setup
- Install Homebrew first → then Docker and gcloud via brew
- Add gcloud to PATH: `export PATH=/opt/homebrew/share/google-cloud-sdk/bin:$PATH`
- Docker Desktop must be running before any docker commands
- Use `gcloud auth application-default login` separately from `gcloud auth login`
- Set quota project: `gcloud auth application-default set-quota-project PROJECT_ID`

### Docker Build
- Always build with `--platform linux/amd64` for Cloud Run (M2 Mac builds ARM64 by default)
- `python:3.11-slim` needs system libs for OpenCV — add to Dockerfile
- Use `libgl1` not `libgl1-mesa-glx` (package name changed in Debian trixie)
- Use `--no-cache` when Dockerfile changes to avoid stale cached layers
- Test locally with `docker run -p 8080:8080` before pushing to GCR

### Cloud Run
- Image must be AMD64 — Cloud Run does not support ARM64
- Use `--memory 4Gi --cpu 2 --timeout 300` for model loading
- Check logs with `gcloud logging read` when deployment fails

### Ultralytics HUB vs Roboflow
- Roboflow does NOT support uploading `.pt` files directly
- Ultralytics HUB is the right platform for YOLO `.pt` models
- Select closest region for lowest latency

### Waymo License
- Waymo Open Dataset is non-commercial only
- Academic/portfolio use is allowed
- Must include license notice in all deployments
